# Was ist neu?

Diese Tabelle zeigt, was in den letzten sieben Tagen in der Deutsche Digitale Bibliothek hinzugekommen ist.

In [1]:
import pandas as pd
import requests
import base64
import hashlib
from urllib.parse import quote
from html import escape
from datetime import datetime, timedelta
from IPython.display import Markdown, HTML, display
from tqdm.auto import tqdm

# ------------------------------------------------------------
# Konfiguration
# ------------------------------------------------------------

SEARCH_URL = "https://api.deutsche-digitale-bibliothek.de/2/search/index/search/select"
ITEM_URL = "https://api.deutsche-digitale-bibliothek.de/2/items/{item_id}/view"

# Zeitraum: volle Tage von START_DATE 00:00:00Z bis END_DATE 23:59:59Z
DAYS_BACK = 7  # x Tage zurück

today = datetime.now().date()
START_DATE = today - timedelta(days=DAYS_BACK)
END_DATE = today + timedelta(days=1)

START_DATE_ISO = f"{START_DATE.isoformat()}T00:00:00Z"
END_DATE_ISO = f"{END_DATE.isoformat()}T00:00:00Z"

# Präfix für die Berechnung der DDB-ID aus supplier_id
SUPPLIER_PREFIX = "www_fiz-karlsruhe_de"

# ------------------------------------------------------------
# Mapping-Tabellen
# ------------------------------------------------------------

PROVIDER_SECTOR_LABELS = {
    "sec_01": "Archiv",
    "sec_02": "Bibliothek",
    "sec_03": "Denkmalpflege",
    "sec_04": "Wissenschaft",
    "sec_05": "Mediathek",
    "sec_06": "Museum",
    "sec_07": "Sonstige",
}

TYPE_FCT_LABELS = {
    "mediatype_001": "Audio",
    "mediatype_002": "Bild",
    "mediatype_003": "Text",
    "mediatype_004": "Volltext",
    "mediatype_005": "Video",
    "mediatype_006": "Sonstige",
    "mediatype_007": "Kein Medientyp",
    "mediatype_008": "Organisation",
}


# ------------------------------------------------------------
# Hilfsfunktionen
# ------------------------------------------------------------

def scalar_or_list(values):
    """
    Wandelt eine Liste passend um.

    []              -> None
    ["A"]           -> "A"
    ["A", "B"]      -> ["A", "B"]

    Dadurch werden einfache Werte nicht unnötig als Liste gespeichert.
    """
    values = [value for value in values if value is not None]

    if len(values) == 0:
        return None

    if len(values) == 1:
        return values[0]

    return values


def as_list(value):
    """
    Macht aus None, Skalar oder Liste immer eine Liste.

    None            -> []
    "A"             -> ["A"]
    ["A", "B"]      -> ["A", "B"]

    Das vereinfacht die Verarbeitung von Mehrfachwerten.
    """
    if value is None:
        return []

    if isinstance(value, list):
        return value

    return [value]


def replace_values(value, mapping):
    """
    Ersetzt Codes durch lesbare Werte.

    Beispiel:
      "sec_02" -> "Bibliothek"

    Funktioniert auch mit Mehrfachwerten:
      ["sec_01", "sec_06"] -> ["Archiv", "Museum"]

    Unbekannte Werte bleiben unverändert.
    """
    values = [
        mapping.get(single_value, single_value)
        for single_value in as_list(value)
    ]

    return scalar_or_list(values)


def facet_values_with_counts(values_and_counts, mapping):
    """
    Wandelt eine Solr-Facette inklusive Counts um.

    Solr liefert:
      ["mediatype_002", 123, "mediatype_003", 45]

    Daraus wird:
      ["Bild (123)", "Text (45)"]

    Bei nur einem Wert wird ein Skalar zurückgegeben:
      "Bild (123)"

    Wichtig:
    Das Mapping wird vor dem Anhängen des Counts angewendet.
    """
    values = []

    for code, count in zip(values_and_counts[0::2], values_and_counts[1::2]):
        label = mapping.get(code, code)
        values.append(f"{label} ({count})")

    return scalar_or_list(values)


def calculate_ddb_id(value):
    """
    Berechnet aus einer ursprünglichen supplier_id die DDB-Item-ID.

    Vorschrift:
      SHA1("www_fiz-karlsruhe_de{supplier_id}")
      BASE32(SHA1-Digest)

    Wichtig:
    Es wird der binäre SHA1-Digest verwendet, nicht der Hex-String.
    """
    text = f"{SUPPLIER_PREFIX}{value}"
    sha1_bytes = hashlib.sha1(text.encode("utf-8")).digest()

    return base64.b32encode(sha1_bytes).decode("ascii")


def calculate_ddb_ids(value):
    """
    Berechnet DDB-IDs für Skalar oder Liste.

    "99900714"          -> "BERECHNETE_ID"
    ["99900714", "123"] -> ["BERECHNETE_ID_1", "BERECHNETE_ID_2"]
    None                -> None
    """
    calculated = [
        calculate_ddb_id(single_value)
        for single_value in as_list(value)
    ]

    return scalar_or_list(calculated)


def join_values(value, separator=", "):
    """
    Macht Skalar- oder Listenwerte als Text nutzbar.

    Listen werden mit dem angegebenen Trennzeichen zusammengefügt.
    """
    if value is None:
        return ""

    if isinstance(value, list):
        return separator.join(str(single_value) for single_value in value)

    return str(value)


# Cache für /items/{id}/view.
# Dadurch wird dieselbe ID nicht mehrfach aus der API geladen.
item_cache = {}


def get_item(item_id):
    """
    Lädt ein Item aus der DDB-API:

      /2/items/{item_id}/view

    Die Antwort wird gecacht.

    Falls ein Item nicht gefunden wird, wird ein leeres Dict zurückgegeben.
    Dadurch bricht das Skript bei einzelnen fehlenden Items nicht komplett ab.
    """
    if item_id in item_cache:
        return item_cache[item_id]

    url = ITEM_URL.format(item_id=quote(str(item_id), safe=""))

    response = requests.get(url)

    if response.status_code == 404:
        item_cache[item_id] = {}
        return item_cache[item_id]

    response.raise_for_status()

    item_cache[item_id] = response.json()
    return item_cache[item_id]


def get_institution_value(item_id_or_ids, field):
    """
    Liest aus /items/{id}/view:

      JSON["cortex-institution"][field]

    Beispiele:
      field = "name"
      field = "sector"

    Funktioniert mit einzelner ID und mit Listen von IDs.
    """
    values = []

    for item_id in as_list(item_id_or_ids):
        item = get_item(item_id)

        value = (
            item
            .get("cortex-institution", {})
            .get(field)
        )

        values.append(value)

    return scalar_or_list(values)


# tqdm für pandas aktivieren
tqdm.pandas()


# ------------------------------------------------------------
# Verarbeitung
# ------------------------------------------------------------

with tqdm(total=12, desc="Gesamtfortschritt", unit="Schritt") as progress:

    # --------------------------------------------------------
    # 1. dataset_id / dataprovider_id der letzten Woche holen
    # --------------------------------------------------------

    params = [
        ("q", "*:*"),
        ("fq", f'last_update:["{START_DATE_ISO}" TO "{END_DATE_ISO}"]'),
        ("fq", "dataset_id:*"),
        ("fq", r"dataprovider_id:/[A-Za-z0-9]{32}/"),
        ("rows", "0"),

        # Pivot-Facette:
        # Erst dataset_id, darunter dataprovider_id.
        ("facet", "true"),
        ("facet.pivot", "dataset_id,dataprovider_id"),
        ("facet.limit", "-1"),
        ("facet.pivot.mincount", "1"),

        # Wichtig:
        # Nicht nur Dokumente filtern, sondern auch die ausgegebenen Facettenwerte.
        ("f.dataprovider_id.facet.matches", r"^[A-Za-z0-9]{32}$"),

        ("wt", "json"),
    ]

    response = requests.get(
        SEARCH_URL,
        params=params
    )
    response.raise_for_status()

    data = response.json()
    progress.update(1)

    # --------------------------------------------------------
    # 2. Pivot-Ergebnis in ein DataFrame schreiben
    # --------------------------------------------------------

    rows = []

    pivots = data["facet_counts"]["facet_pivot"]["dataset_id,dataprovider_id"]

    for dataset in tqdm(
        pivots,
        desc="Pivot-Ergebnis verarbeiten",
        unit="Dataset",
        leave=False,
    ):
        dataset_id = dataset["value"]

        for provider in dataset.get("pivot", []):
            rows.append({
                "dataset_id": dataset_id,
                "dataprovider_id": provider["value"],
                "count": provider["count"],
            })

    df = pd.DataFrame(rows)

    if df.empty:
        raise SystemExit("Keine Treffer gefunden.")

    progress.update(1)

    # --------------------------------------------------------
    # 3. Zusatzdaten je dataset_id holen
    # --------------------------------------------------------

    metadata_rows = []
    dataset_ids = sorted(df["dataset_id"].dropna().unique())

    for dataset_id in tqdm(
        dataset_ids,
        desc="Zusatzdaten je dataset_id laden",
        unit="Dataset",
        leave=False,
    ):
        params = [
            ("q", f'dataset_id:"{dataset_id}"'),
            ("rows", "0"),

            # Facetten für Zusatzinformationen
            ("facet", "true"),
            ("facet.mincount", "1"),
            ("facet.limit", "-1"),
            ("facet.field", "md_format"),
            ("facet.field", "type_fct"),
            ("facet.field", "supplier_id"),
            ("facet.field", "dataset_label"),

            ("wt", "json"),
        ]

        response = requests.get(
            SEARCH_URL,
            params=params
        )
        response.raise_for_status()

        metadata = response.json()
        facet_fields = metadata["facet_counts"]["facet_fields"]

        row = {
            "dataset_id": dataset_id,
        }

        # Normale Facetten:
        # Solr liefert:
        # ["Wert 1", Count 1, "Wert 2", Count 2, ...]
        #
        # Für diese Felder brauchen wir nur die Werte.
        for field in ["md_format", "supplier_id", "dataset_label"]:
            values = facet_fields.get(field, [])[0::2]
            row[field] = scalar_or_list(values)

        # type_fct:
        # Hier sollen Wert und Count erhalten bleiben.
        #
        # Beispiel:
        # ["mediatype_002", 123, "mediatype_003", 45]
        #
        # Ergebnis:
        # ["Bild (123)", "Text (45)"]
        type_fct_values_and_counts = facet_fields.get("type_fct", [])
        row["type_fct"] = join_values(
            facet_values_with_counts(
                type_fct_values_and_counts,
                TYPE_FCT_LABELS,
            ),
            separator=", ",
        )

        metadata_rows.append(row)

    metadata_df = pd.DataFrame(metadata_rows)
    progress.update(1)

    # --------------------------------------------------------
    # 4. Hauptdaten und Zusatzdaten zusammenführen
    # --------------------------------------------------------

    df = df.merge(
        metadata_df,
        on="dataset_id",
        how="left",
    )

    progress.update(1)

    # --------------------------------------------------------
    # 5. supplier_id berechnen und ursprüngliche Werte ersetzen
    # --------------------------------------------------------

    # Die supplier_id aus Solr ist z. B. "99900714".
    #
    # Für /items/{supplier_id}/view brauchen wir aber die berechnete DDB-ID.
    # Deshalb wird supplier_id hier bewusst überschrieben.
    df["supplier_id"] = df["supplier_id"].progress_apply(calculate_ddb_ids)

    progress.update(1)

    # --------------------------------------------------------
    # 6. Provider-Namen laden
    # --------------------------------------------------------

    # Quelle:
    # /2/items/{dataprovider_id}/view
    # JSON["cortex-institution"]["name"]
    df["provider_name"] = df["dataprovider_id"].progress_apply(
        lambda value: get_institution_value(value, "name")
    )

    progress.update(1)

    # --------------------------------------------------------
    # 7. Provider-Sektoren laden
    # --------------------------------------------------------

    # Quelle:
    # /2/items/{dataprovider_id}/view
    # JSON["cortex-institution"]["sector"]
    df["provider_sector"] = df["dataprovider_id"].progress_apply(
        lambda value: get_institution_value(value, "sector")
    )

    progress.update(1)

    # --------------------------------------------------------
    # 8. Supplier-Namen laden
    # --------------------------------------------------------

    # Quelle:
    # /2/items/{supplier_id}/view
    # JSON["cortex-institution"]["name"]
    #
    # supplier_id ist hier bereits die berechnete DDB-ID.
    df["supplier_name"] = df["supplier_id"].progress_apply(
        lambda value: get_institution_value(value, "name")
    )

    progress.update(1)

    # --------------------------------------------------------
    # 9. Codes durch lesbare Bezeichnungen ersetzen
    # --------------------------------------------------------

    # provider_sector enthält Werte wie sec_02.
    df["provider_sector"] = df["provider_sector"].progress_apply(
        lambda value: replace_values(value, PROVIDER_SECTOR_LABELS)
    )

    # type_fct wurde bereits beim Auslesen der Facette ersetzt,
    # weil dort zusätzlich der Count angehängt wird.
    progress.update(1)

    # --------------------------------------------------------
    # 10. Spalten sortieren
    # --------------------------------------------------------

    # count und type_fct sind hier bewusst vertauscht:
    # count steht vor type_fct.
    df = df[
        [
            "dataprovider_id",
            "provider_name",
            "provider_sector",

            "md_format",
            "count",
            "type_fct",

            "dataset_id",
            "dataset_label",

            "supplier_id",
            "supplier_name",
        ]
    ]

    progress.update(1)

    # --------------------------------------------------------
    # 11. Zeilen sortieren
    # --------------------------------------------------------

    # Manche Spalten können intern Listen enthalten.
    # Deshalb werden für die Sortierung temporäre Textspalten erzeugt.
    df["_sort_provider_sector"] = df["provider_sector"].progress_apply(lambda value: join_values(value, separator="; "))
    df["_sort_provider_name"] = df["provider_name"].progress_apply(lambda value: join_values(value, separator="; "))

    df = df.sort_values(
        by=["_sort_provider_sector", "_sort_provider_name", "count"],
        ascending=[True, True, False],
    ).drop(
        columns=["_sort_provider_sector", "_sort_provider_name"]
    ).reset_index(drop=True)

    progress.update(1)

    # --------------------------------------------------------
    # 12. dataset_id verlinken
    # --------------------------------------------------------


    df["dataset_id"] = df["dataset_id"].apply(
        lambda id: (
            "" if pd.isna(id) or id == ""
            else f'<a href="https://www.deutsche-digitale-bibliothek.de/searchresults?query={quote(f"dataset_id:{id}")}" target="_blank">{escape(str(id))}</a>'
        )
    )

    df["supplier_name"] = df.apply(
        lambda row: (
            "" if pd.isna(row["supplier_id"]) or pd.isna(row["supplier_name"])
            else (
                f'<a href="https://www.deutsche-digitale-bibliothek.de/organization/{quote(str(row["supplier_id"]))}" '
                f'target="_blank">{escape(str(row["supplier_name"]))}</a>'
            )
        ),
        axis=1
    )

    df["provider_name"] = df.apply(
        lambda row: (
            "" if pd.isna(row["dataprovider_id"]) or pd.isna(row["provider_name"])
            else (
                f'<a href="https://www.deutsche-digitale-bibliothek.de/organization/{quote(str(row["dataprovider_id"]))}" '
                f'target="_blank">{escape(str(row["provider_name"]))}</a>'
            )
        ),
        axis=1
    )

    progress.update(1)


# ------------------------------------------------------------
# Ergebnis anzeigen
# ------------------------------------------------------------

# Stand: Datum/Uhrzeit der Notebook-Ausführung (lokale Zeitzone)
stand = datetime.now().astimezone().strftime("%d.%m.%Y um %H:%M:%S Uhr")
display(Markdown(f"**Letzte Aktualisierung:** {stand}"))
display(Markdown(f"**Zeitraum:** {START_DATE.strftime('%d.%m.%Y')} bis {END_DATE.strftime('%d.%m.%Y')}"))

# In Jupyter/Notebook:
display(HTML(df.drop(columns=["supplier_id", "dataprovider_id"], errors="ignore").to_html(escape=False, index=False)))

/opt/hostedtoolcache/Python/3.12.14/x64/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Gesamtfortschritt:   0%|          | 0/12 [00:00<?, ?Schritt/s]

Gesamtfortschritt:   8%|▊         | 1/12 [00:04<00:50,  4.63s/Schritt]

Pivot-Ergebnis verarbeiten:   0%|          | 0/14 [00:00<?, ?Dataset/s]

Zusatzdaten je dataset_id laden:   0%|          | 0/14 [00:00<?, ?Dataset/s]

...

100%|██████████| 14/14 [00:00<00:00, 54421.00it/s]


Gesamtfortschritt: 100%|██████████| 12/12 [00:27<00:00,  2.28s/Schritt]

**Letzte Aktualisierung:** 27.08.2026 um 14:05:40 Uhr

**Zeitraum:** 20.08.2026 bis 28.08.2026

provider_name,provider_sector,md_format,count,type_fct,dataset_id,dataset_label,supplier_name
Historisches Archiv des Germanischen Nationalmuseums,Archiv,ead,335,Kein Medientyp (335),30928396577133543pzRK,Gesamtlieferung (Tektonik) - EAD,Historisches Archiv des Germanischen Nationalmuseums
Landeskirchliches Archiv der Evangelisch-Lutherischen Kirche in Norddeutschland,Archiv,ead,52030,Kein Medientyp (52030),18922344786108738DJZT,Gesamtlieferung (Findbuch) - EAD,Landeskirchliches Archiv der Evangelisch-Lutherischen Kirche in Norddeutschland
Stadt-und Hospitalarchiv Schwäbisch Hall,Archiv,ead,19616,Kein Medientyp (19616),462572956552718sSTJ,Gesamtlieferung (Findbuch) - EAD,Stadt-und Hospitalarchiv Schwäbisch Hall
Niedersächsische Staats- und Universitätsbibliothek Göttingen,Bibliothek,mets,80321,"Bild (170), Text (80151)",447386867306790qpRt,Hauptportal: 1843g - EU - SUB Göttingen - (00012417) - METS/MODS,Niedersächsische Staats- und Universitätsbibliothek Göttingen
Ruprecht-Karls-Universität Heidelberg. Universitätsbibliothek,Bibliothek,lido,23631,Bild (23631),482177567439236yBog,Gesamtlieferung: UB Sontheimer,Ruprecht-Karls-Universität Heidelberg. Universitätsbibliothek
Ruprecht-Karls-Universität Heidelberg. Universitätsbibliothek,Bibliothek,lido,3005,"Bild (3004), Kein Medientyp (1)",478514360817945ubMY,Gesamtlieferung: UB Antikensammlung (heiColl) - LIDO,Ruprecht-Karls-Universität Heidelberg. Universitätsbibliothek
Ruprecht-Karls-Universität Heidelberg. Universitätsbibliothek,Bibliothek,lido,680,Bild (680),480003086043916Wtup,Gesamtlieferung: UB Sammlung Fehr (heiColl) - LIDO,Ruprecht-Karls-Universität Heidelberg. Universitätsbibliothek
Ruprecht-Karls-Universität Heidelberg. Universitätsbibliothek,Bibliothek,lido,413,Bild (413),478673245334749XlgO,Gesamtlieferung: UB HiReiBeSAI (heiColl) - LIDO,Ruprecht-Karls-Universität Heidelberg. Universitätsbibliothek
Ruprecht-Karls-Universität Heidelberg. Universitätsbibliothek,Bibliothek,lido,32,Bild (32),480706741712647UrtF,Gesamtlieferung: UB EXC XingXing (heiColl) - LIDO,Ruprecht-Karls-Universität Heidelberg. Universitätsbibliothek
Deutsches Stuhlbaumuseum,Museum,lido,238,Bild (238),1168700575208128IcEi,Deutsches Stuhlbaumuseum (museum-digital) - LIDO,Deutsches Stuhlbaumuseum
